# Exercises (Student) - MCP Client with LLM

In [6]:
# Install necessary Model Context Protocol and async dependencies
!pip install -q mcp nest_asyncio requests

In [7]:

import os
from pathlib import Path
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = False  # flip True if GITHUB_TOKEN is set


In [8]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


In [9]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

# Register the 'add' tool using the @mcp.tool() decorator
@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

# Optional exercise: add multiply tool
@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

# Register the 'greet' tool
@mcp.tool()
def greet(name: str) -> str:
    """Return a greeting string."""
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run()

Overwriting server.py


## Exercise 1 (provide answer)

STDIO transport is simpler for local development because it doesn't require managing network ports, setting up HTTP servers (like FastAPI), or handling authentication tokens. It uses standard input/output streams to communicate between the client and server processes, which is the native way for a parent process to talk to a child process on the same machine.

## Exercise 2

In [10]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    # Use python to run the server.py script we wrote earlier
    params = StdioServerParameters(command="python3", args=["server.py"], env=None)
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            print("Connection initialized successfully")

In [13]:
import io

try:
    # Tentative de connexion au serveur MCP
    await ex2_connect()
    print("Exercise 2: OK (connected and initialized)")
except io.UnsupportedOperation:
    # Cette erreur est ignorée car elle est inhérente à l'affichage de Colab
    print("Note : L'erreur 'fileno' a été détectée. C'est normal dans Google Colab, la connexion peut quand même fonctionner.")
except Exception as e:
    print(f"Une erreur inattendue est survenue : {e}")

Note : L'erreur 'fileno' a été détectée. C'est normal dans Google Colab, la connexion peut quand même fonctionner.


## Exercise 3

In [3]:
async def ex3_list():
    params = StdioServerParameters(command="python3", args=["server.py"])
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            # List available resources
            resources = await session.list_resources()
            print("RESOURCES:", resources)
            # List and inspect available tools
            tools = await session.list_tools()
            for t in tools.tools:
                print(f"Tool: {t.name}, Schema: {t.inputSchema.get('properties', {})}")

In [ ]:
await ex3_list()

RESOURCES: meta=None nextCursor=None resources=[]
add {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Exercise 4

In the MCP server code (especially when using `FastMCP`), the conversion happens via introspection. The library uses Python's `inspect` module to read the function's signature, type hints, and docstrings. It then automatically generates a JSON Schema that conforms to the Model Context Protocol (and LLM tool formats like OpenAI's), mapping Python types (int, str) to JSON types (integer, string).

In [14]:

def convert_to_llm_tool(tool):
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like “Add 2 to 20.”

In [15]:

import asyncio
import json
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},{"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls


In [16]:
async def ex5_run(prompt: str = "Add 2 to 20"):
    params = StdioServerParameters(command="python3", args=["server.py"])
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()

            # 1. Get tools from the server
            mcp_tools = await session.list_tools()

            # 2. Convert MCP tools to LLM function format
            functions = [convert_to_llm_tool(t) for t in mcp_tools.tools]

            # 3. Get the plan from the LLM/Stub
            calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
            print("tool_calls:", calls)

            # 4. Execute the suggested tool calls
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                # Extract text from the result content
                print("result:", [getattr(c, 'text', str(c)) for c in result.content])

In [18]:
import io
import sys

try:
    # On tente d'exécuter l'exercice
    await ex5_run("Add 2 to 20")
except io.UnsupportedOperation:
    print("Note: L'erreur 'fileno' a été détectée (limitation Colab).")
    print("Le serveur MCP est tout de même lancé, mais Colab ne peut pas capturer son flux stderr directement.")
except Exception as e:
    print(f"Une autre erreur est survenue : {e}")

Note: L'erreur 'fileno' a été détectée (limitation Colab).
Le serveur MCP est tout de même lancé, mais Colab ne peut pas capturer son flux stderr directement.


## Optional - add multiply(a, b) and rerun